# 01 · Khám phá dữ liệu (EDA)

Train có đáp án; public chỉ có câu hỏi, nên phần public chỉ là số đếm. Output chứa dữ liệu thi: **xóa output trước khi commit**
(`jupyter nbconvert --clear-output --inplace notebooks/*.ipynb`).

In [ ]:
import json, re, sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
train_q, public_q = read(DATA / "train/questions.jsonl"), read(DATA / "public_test/questions.jsonl")
train_a = {a["id"]: a for a in read(DATA / "train/answers.jsonl")}
splits = {"train": train_q, "public": public_q}
pd.DataFrame({name: {"questions": len(qs), "shops": len({q["business_id"] for q in qs}),
                     "questions per shop (min / median / max)": "{} / {:.0f} / {}".format(*(lambda c: (min(c), pd.Series(c).median(), max(c)))(list(Counter(q["business_id"] for q in qs).values()))),
                     "as_of_date range": f'{min(q["as_of_date"] for q in qs)} .. {max(q["as_of_date"] for q in qs)}'}
              for name, qs in splits.items()})

## Tệp dữ liệu của shop

Mỗi shop là một thư mục gồm README.md và các bảng (csv / xlsx / sqlite); số bảng và dung lượng khác nhau rất nhiều.

In [ ]:
rows = []
for split, folder in (("train", "train"), ("public", "public_test")):
    for shop in sorted((DATA / folder / "data").iterdir()):
        files = [f for f in shop.rglob("*") if f.is_file() and f.name != "README.md"]
        rows.append({"split": split, "shop": shop.name, "files": len(files), "MB": round(sum(f.stat().st_size for f in files) / 2**20, 1),
                     **Counter(f.suffix.lower().lstrip(".") for f in files)})
files = pd.DataFrame(rows).fillna(0)
files.groupby("split")[["files", "MB"] + [c for c in files.columns if c not in ("split", "shop", "files", "MB")]].sum()

In [ ]:
files.groupby("split")[["files", "MB"]].describe().round(1)

## Loại đáp án (train)

Phân bố type của đáp án chuẩn; `alias` là các cách viết khác được chấp nhận (mã hàng, cách viết khác trong dữ liệu).

In [ ]:
types = Counter(a["type"] for a in train_a.values())
ax = pd.Series(types).sort_values().plot.barh(figsize=(6, 3), title="train answer types")
print({t: f"{n / len(train_a):.1%}" for t, n in types.most_common()})
print("answers with aliases:", sum(bool(a.get("alias")) for a in train_a.values()))

## Dạng câu hỏi

Tỷ lệ câu kích hoạt từng gợi ý trong `datamate/hints.py` và `datamate/prompts.py`. Dạng câu có trong cả train và public thì kiểm chứng được
bằng đáp án train; dạng chỉ có ở public (tiền viết tắt, từ địa phương) thì không.

In [ ]:
from datamate import hints as H
from datamate.ingest import norm
from datamate.prompts import SLANG, money_gloss
shapes = {
    "period named": lambda q: bool(H.period_hints(q["question"], q["as_of_date"])),
    "debt": lambda q: bool(H.DEBT_WORDS.search(norm(q["question"]))),
    "days by daily total": lambda q: bool(H.COUNT_DAYS.search(norm(q["question"])) and H.THRESHOLD.search(norm(q["question"]) + q["question"])),
    "profit": lambda q: bool(H.PROFIT.search(norm(q["question"]))),
    "delivery trips": lambda q: bool(H.TRIPS.search(norm(q["question"]))),
    "lots / expiry": lambda q: bool(H.LOT.search(norm(q["question"]))),
    "money shorthand": lambda q: bool(money_gloss(q["question"])),
    "chat words": lambda q: bool(set(re.findall(r"\w+", q["question"].lower())) & set(SLANG)),
}
share = pd.DataFrame({name: {k: sum(f(q) for q in qs) / len(qs) for k, f in shapes.items()} for name, qs in splits.items()})
share.plot.barh(figsize=(7, 4), title="share of questions"); share.style.format("{:.1%}")

In [ ]:
pd.DataFrame({name: pd.Series([len(q["question"]) for q in qs]).describe().round(0) for name, qs in splits.items()}).T

## Rulebook

Train và public có rulebook riêng; public thêm một số luật (khách trung thành, giờ cao điểm, nhà cung cấp chính…).

In [ ]:
rb = {s: (DATA / f / "rulebook.md").read_text(encoding="utf-8") for s, f in (("train", "train"), ("public", "public_test"))}
rules = {s: {l.split("|")[1].strip() for l in t.splitlines() if l.startswith("| ") and not l.startswith("| Luật")} for s, t in rb.items()}
print(len(rules["train"]), "rules in train,", len(rules["public"]), "in public")
print("only in public:", sorted(rules["public"] - rules["train"]))
print("only in train:", sorted(rules["train"] - rules["public"]))

## Một shop sau khi nạp

`load_business` đọc mọi bảng vào DuckDB; `schema_text` là phần SCHEMA mà model nhìn thấy.

In [ ]:
from datamate.ingest import load_business, schema_text
db, meta = load_business(DATA / "train", "biz_101", canon=True)
display(pd.DataFrame([{"table": t["name"], "rows": t["rows"], "columns": len(t["columns"])} for t in meta["tables"]]))
print(schema_text(meta)[:3000])